In [1]:
import os

In [2]:
%pwd

'c:\\Projects\\End-to-End-ML-Classification-Project-with-MLflow-\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Projects\\End-to-End-ML-Classification-Project-with-MLflow-'

In [ ]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    preprocessor: str
    target_column: str

In [6]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        schema_target =  self.schema.TARGET_COLUMN
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            preprocessor=config.preprocessor_name,
            target_column = schema_target.name
            all_schema=schema
#            train_preprocessor=config.train_preprocessor_name,
        )

        return data_transformation_config

In [ ]:
import os
from mlProject import logger
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler, FunctionTransformer
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import pandas as pd
import joblib

In [ ]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
    
    
    ## Note: You can add different data transformation techniques such as Scaler, PCA and all
    #You can perform all kinds of EDA in ML cycle here before passing this data to the model

    
    # Converting objects into numeric
    def coerce_numeric_func(self, X):
        return X.apply(pd.to_numeric, errors="coerce")




    def get_data_transformer_obj(self):

        """"
        This function transforms the the dataset.
        
        """

        numeric_columns, categorical_columns, mixed_columns = [], [], []

        for col, dtype in self.config.all_schema.items():
            if dtype=="number": # type: ignore
                numeric_columns.append(col)
            elif dtype=="object":
                categorical_columns.append(col)
            else:
                mixed_columns.append(col)

        

        num_pipeline= SkPipeline(
                steps=[
                ("imputer",SimpleImputer(strategy="median")),
                ("scaler",StandardScaler())
                ]
            )

        cat_pipeline=SkPipeline(
                steps=[
                ("imputer",SimpleImputer(strategy="most_frequent")),
                ("one_hot_encoder",OneHotEncoder(handle_unknown='ignore')),
                ("scaler",StandardScaler(with_mean=False))
                ]
            )


        numeric_coercer = FunctionTransformer(
                self.coerce_numeric_func,
                feature_names_out="one-to-one"
            )

        mixed_pipeline = SkPipeline(
                steps=[
                ("coerce_numeric", numeric_coercer),
                ("imputer",SimpleImputer(strategy="mean")),
                #("one_hot_encoder", OneHotEncoder(handle_unknown='ignore')),
                ("scaler", StandardScaler(with_mean=False))
                #("target_encoder",TargetEncoder(smoothing=10, min_samples_leaf=5))
                ]
            )

        # ColumnTransformer
        preprocessor=ColumnTransformer(
                [
                ("num_pipeline",num_pipeline,numeric_columns),
                ("cat_pipelines",cat_pipeline,categorical_columns),
                ("mixed_pipeline", mixed_pipeline, mixed_columns)
                ]
            )
        
#        # Smote Pipeline
#        train_data_preprocessor = Pipeline(
#            [
#                ("preprocessor", test_data_preprocessor),
#                ("smote", SMOTE())
#            ]
#        )

#        joblib.dump(train_data_preprocessor, os.path.join(self.config.root_dir, self.config.train_preprocessor_name))

        return preprocessor



    def initiate_data_transformation(self):
        data = pd.read_excel(self.config.data_path)

        # Split the data into training and test sets. (0.75, 0.25) split.
        train, test = train_test_split(data, test_size=0.25, random_state=1)


        logger.info("Splitted data into training and test sets")
        logger.info(train.shape)
        logger.info(test.shape)

        print(train.shape)
        print(test.shape)

        train_x = train.drop([self.config.target_column], axis=1)
        test_x = test.drop([self.config.target_column], axis=1)
        train_y = train[self.config.target_column]
        test_y = test[self.config.target_column]

        logger.info(f"Applying preprocessing object on training dataframe and testing dataframe.")

        preprocessor = self.get_data_transformer_obj()

        train_x = preprocessor.fit_transform(train_x)
        test_x = preprocessor.transform(test_x)

#        train_x.to_csv(os.path.join(self.config.root_dir, "train.csv"),index = False)
#        test_x.to_csv(os.path.join(self.config.root_dir, "test.csv"),index = False)

        joblib.dump(preprocessor, os.path.join(self.config.root_dir, self.config.preprocessor))

        return train_x, test_x, train_y, test_y

        
        
        

In [ ]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.initiate_data_transformation()
except Exception as e:
    raise e

[2023-07-11 10:52:42,375: INFO: common: yaml file: config\config.yaml loaded successfully]
[2023-07-11 10:52:42,377: INFO: common: yaml file: params.yaml loaded successfully]
[2023-07-11 10:52:42,379: INFO: common: yaml file: schema.yaml loaded successfully]
[2023-07-11 10:52:42,380: INFO: common: created directory at: artifacts]
[2023-07-11 10:52:42,381: INFO: common: created directory at: artifacts/data_transformation]
[2023-07-11 10:52:42,406: INFO: 741865018: Splited data into training and test sets]
[2023-07-11 10:52:42,407: INFO: 741865018: (1199, 12)]
[2023-07-11 10:52:42,407: INFO: 741865018: (400, 12)]
(1199, 12)
(400, 12)
